In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
CREATE TABLE IF NOT EXISTS ML_BENCHMARK_RESULTS (
    MODEL_CLASS STRING,
    COMPUTE_POOL STRING,
    RUN_ID INT,
    N_COLS_SAMPLED INT,
    N_ROWS_SAMPLED INT,
    DURATION_SECONDS FLOAT,
    START_TIMESTAMP FLOAT
);

In [ ]:
CREATE OR REPLACE STAGE PAYLOAD_STAGE;

In [ ]:
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_XS_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_XS;
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_S_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_S;
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_M_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_M;
CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_SL_TEST
  AUTO_RESUME = TRUE 
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_SL;
-- CREATE COMPUTE POOL IF NOT EXISTS CPU_X64_L_TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = CPU_X64_L;
-- CREATE COMPUTE POOL IF NOT EXISTS HIGHMEM_X64_S_TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = HIGHMEM_X64_S;
-- CREATE COMPUTE POOL IF NOT EXISTS HIGHMEM_X64_M_TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = HIGHMEM_X64_M;
-- CREATE COMPUTE POOL IF NOT EXISTS HIGHMEM_X64_L _TEST
--   AUTO_RESUME = TRUE 
--   INITALLY_SUSPENDED = TRUE
--   MIN_NODES = 1
--   MAX_NODES = 1
--   INSTANCE_FAMILY = HIGHMEM_X64_L;

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
import sys
import pandas as pd

# --- Configuration ---
N_SAMPLES = 1_000_000  # 5 million rows
N_FEATURES = 100       # 100 columns (features)
N_CLASSES = 2          # Binary classification

# --- 1. Data Generation ---
print("Starting data generation...")
# Generate the feature matrix (X) and the target vector (y)
X_full, y_full = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=80,      # 80 useful features (high correlation with target)
    n_redundant=10,        # 10 features that are combinations of informative ones
    n_classes=N_CLASSES,   # Binary classification
    random_state=42,       # For reproducibility
    shuffle=True,
)
print("✅ Data generation complete.")

# --- 2. Memory Analysis ---
# Calculate the memory size of the X matrix in GB
x_memory_gb = X_full.nbytes / (1024**3)

print("\n--- Dataset Summary ---")
print(f"X Shape: {X_full.shape} (Features)")
print(f"y Shape: {y_full.shape} (Labels)")
print(f"X Data Type: {X_full.dtype} ({X_full.dtype.itemsize} bytes per value)")
print(f"Estimated X Memory Footprint: {x_memory_gb:.2f} GB")

In [ ]:
# --- NEW Step 3. Convert and Save to Snowflake Table ---
print("Starting conversion to Snowpark DataFrame and saving to Snowflake...")

# 3a. Combine X and y into a single Pandas DataFrame for easy conversion
# Create feature column names F0, F1, F2, ...
feature_cols = [f'F{i}' for i in range(X_full.shape[1])]
df_combined_pandas = pd.DataFrame(X_full, columns=feature_cols)
df_combined_pandas['TARGET'] = y_full

# 3b. Convert Pandas DataFrame to Snowpark DataFrame
df_snowpark = session.create_dataframe(df_combined_pandas)

# 3c. Save the data to a new Snowflake table
DATA_TABLE_NAME = "BENCHMARK_RAW_DATA"
df_snowpark.write.mode("overwrite").save_as_table(DATA_TABLE_NAME)

print(f"✅ Data saved to Snowflake table: {DATA_TABLE_NAME}")

# --- Set the table name as a global variable needed later ---
# Now the data source is the Snowflake table, not the local NumPy array
# X_full and y_full are no longer needed locally and can be deleted to save memory
del X_full
del y_full

In [ ]:
# from sklearn.ensemble import RandomForestClassifier

# # Define the Random Forest Classifier with balanced hyperparameters
# rf_base_estimator = RandomForestClassifier(
#     # --- Ensemble Structure ---
#     n_estimators=200,          # The number of trees in the forest. More trees usually means better
#                                # performance but takes longer. 200 is a good starting point.
    
#     # --- Tree Depth and Splitting ---
#     max_depth=20,              # The maximum depth of the tree. Limits overfitting by preventing
#                                # trees from becoming too complex. A value like 15-20 is moderate.
#     min_samples_leaf=5,        # The minimum number of samples required to be at a leaf node.
#                                # Prevents overfitting to rare patterns.
    
#     # --- Feature Sampling ---
#     max_features='sqrt',       # The number of features to consider when looking for the best split.
#                                # 'sqrt' (square root of total features) is the recommended default
#                                # for classification, balancing randomness and signal.
#                                # 
#     # --- General ---
#     n_jobs=-1,                 # Use all available processors for computation (fast training).
#     random_state=42,           # Ensures reproducibility of the results.
#     verbose=0                  # Suppress logging output during training.
# )

# print(f"Random Forest Estimator configured and ready to train.")

# base_estimators_lst = [rf_base_estimator] 


In [ ]:
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    AdaBoostClassifier,
    VotingClassifier # Useful for combining models later
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Import external libraries
import lightgbm as lgb
import xgboost as xgb
# import torch # Include this if you plan to use PyTorch models

# --- 1. Scikit-learn Ensemble Models (Tree-Based) ---
rf_base_estimator = RandomForestClassifier(
    n_estimators=200,          # Good number of trees
    max_depth=20,              # Moderate depth to prevent deep overfitting
    min_samples_leaf=5,        # Minimum samples per leaf
    max_features='sqrt',       # Recommended for classification
    n_jobs=-1,                 # Use all cores
    random_state=42,           # Reproducibility
)

gb_base_estimator = GradientBoostingClassifier(
    n_estimators=100,          # Number of boosting stages
    learning_rate=0.1,         # Step size shrinkage
    max_depth=3,               # Default is a good starting point
    subsample=0.8,             # Fraction of samples to be used for fitting the individual base learners
    random_state=42,
)

ada_base_estimator = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # Typically uses a shallow stump
    n_estimators=50,
    learning_rate=1.0,
    random_state=42,
)

# --- 2. Scikit-learn Linear and Kernel Models ---
lr_base_estimator = LogisticRegression(
    penalty='l2',              # Regularization type
    C=1.0,                     # Inverse of regularization strength
    solver='liblinear',        # Good for small datasets
    max_iter=1000,             # Ensure convergence
    random_state=42,
    n_jobs=-1,
)

svm_base_estimator = SVC(
    C=1.0,                     # Regularization parameter
    kernel='rbf',              # Radial Basis Function kernel
    gamma='scale',             # Kernel coefficient auto-adjusted
    probability=True,          # Enable probability estimates (needed for some ensembles)
    random_state=42,
)

# --- 3. Scikit-learn Simple Models ---
knn_base_estimator = KNeighborsClassifier(
    n_neighbors=5,             # Number of neighbors to use
    weights='uniform',         # All points in the neighborhood are weighted equally
    n_jobs=-1,
)

nb_base_estimator = GaussianNB() # Simple and fast, no major hyperparameters

# --- 4. High-Performance Gradient Boosting Frameworks ---
# LightGBM Classifier
lgbm_base_estimator = lgb.LGBMClassifier(
    n_estimators=150,          # Number of boosting stages
    learning_rate=0.05,        # Slower learning rate for better generalization
    num_leaves=31,             # Controls complexity (main parameter)
    max_depth=-1,              # No limit (let num_leaves control complexity)
    n_jobs=-1,
    random_state=42,
    verbose=-1,                # Suppress output
)

# XGBoost Classifier
xgb_base_estimator = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,               # Control complexity
    subsample=0.8,             # Fraction of samples
    colsample_bytree=0.8,      # Fraction of features
    use_label_encoder=False,   # Recommended setting for newer versions
    eval_metric='logloss',     # Specify evaluation metric
    n_jobs=-1,
    random_state=42,
)

# --- 5. PyTorch Placeholder ---
# Note: A full PyTorch model requires defining an 'nn.Module' class and a training loop.
# This placeholder shows how you might reference it later.
# pytorch_model_placeholder = "Need to define PyTorch_Net class and a wrapper"


base_estimators_lst = [
#    rf_base_estimator,
    gb_base_estimator,
    ada_base_estimator,
    lr_base_estimator,
    svm_base_estimator,
    knn_base_estimator,
    nb_base_estimator,
    lgbm_base_estimator,
    xgb_base_estimator,
    # pytorch_model_placeholder, # Uncomment if you define a wrapper for PyTorch
]

print(f"Total of {len(base_estimators_lst)} estimators configured and ready for training/evaluation.")

In [ ]:
# --- Configuration (Local Script) ---
# Assuming 'session' and 'rf_base_estimator' are defined elsewhere

DATA_TABLE_NAME = "BENCHMARK_RAW_DATA"
RESULTS_TABLE_NAME = "ML_BENCHMARK_RESULTS"
NUM_TOTAL_FEATURES = 100 
# Define the schema for logging results to Snowflake (MUST match the tuple order)
RESULTS_SCHEMA = ['MODEL_TYPE', 'COMPUTE_POOL', 'RUN_ID', 'N_COLS_SAMPLED', 'N_ROWS_SAMPLED', 'DURATION_SECONDS', 'START_TIMESTAMP']

# Lists for combinations
data_cols_lst = list(range(25, 101, 25))
data_rows_lst = list(range(50000, 1000001, 200000))
comp_pool_lst = ["CPU_X64_XS_TEST", "CPU_X64_S_TEST", "CPU_X64_M_TEST", "CPU_X64_SL_TEST"] # Example with multiple pools
runs_lst = [5]


In [ ]:
import itertools
import time
import numpy as np
import pandas as pd
from snowflake.ml.jobs import remote
from snowflake.snowpark import Session 
import concurrent.futures
from threading import Thread

# --------------------------------------------------------------------------
# --- Parallel Pools, Sequential Jobs Within Each Pool ---
print("--- Starting Pool-Parallel, Job-Sequential Benchmark Submissions ---")

# Group combinations by compute pool for parallel execution
combinations_by_pool = {}
for combination in itertools.product(base_estimators_lst, data_cols_lst, data_rows_lst, comp_pool_lst, runs_lst):
    base_estimator_instance, num_cols_to_use, num_rows_to_use, current_pool, runs_to_use = combination
    
    if current_pool not in combinations_by_pool:
        combinations_by_pool[current_pool] = []
    combinations_by_pool[current_pool].append(combination)

total_jobs_in_set = sum(len(combos) for combos in combinations_by_pool.values())
total_jobs_submitted = 0

def process_pool_jobs_sequentially(pool_name, pool_combinations):
    """Process jobs for a specific compute pool ONE AT A TIME"""
    global total_jobs_submitted
    
    print(f"\n🚀 Starting SEQUENTIAL jobs for compute pool: {pool_name}")
    
    completed_jobs = 0
    failed_jobs = 0
    
    # Process each job in the pool ONE BY ONE
    for combination in pool_combinations:
        base_estimator_instance, num_cols_to_use, num_rows_to_use, current_pool, runs_to_use = combination
        
        # Dynamic Inference of Model Components
        MODEL_TYPE_CURRENT = base_estimator_instance.__class__.__name__
        ESTIMATOR_CLASS = base_estimator_instance.__class__
        ESTIMATOR_PARAMS = base_estimator_instance.get_params()

        # Define the remote function for this specific pool
        @remote(current_pool, stage_name="PAYLOAD_STAGE", session=session)
        def ml_run_remote_job(model_type, compute_pool, base_estimator_class, base_estimator_params, num_cols_to_use, num_rows_to_use, runs_to_use, data_table_name, total_features, results_table_name):
            
            session = Session.get_active_session() 
            
            # Data Loading (Accessing staged data from the Snowflake table)
            df_raw = session.table(data_table_name)
            target_col = 'TARGET' 
            feature_cols = [col for col in df_raw.columns if col != target_col]
            df_pd = df_raw.to_pandas()
            X_full = df_pd[feature_cols].to_numpy()
            y_full = df_pd[target_col].to_numpy()
            num_total_features = total_features
            
            # Iterating Runs and Logging Results
            for i in range(0, runs_to_use):
                
                # Sampling and Training setup
                row_indices = np.random.choice(X_full.shape[0], size=num_rows_to_use, replace=False)
                X_rows_sampled = X_full[row_indices, :]
                y_rows_sampled = y_full[row_indices]
                col_indices = np.random.choice(num_total_features, size=num_cols_to_use, replace=False)
                
                X = X_rows_sampled[:, col_indices]
                y = y_rows_sampled
                
                run_estimator = base_estimator_class(**base_estimator_params)
                
                start_time = time.time()
                try:
                    run_estimator.fit(X, y) 
                    end_time = time.time()
                    duration_time = end_time - start_time
                except Exception as e:
                    # Log failed runs with -1 duration
                    end_time = time.time()
                    duration_time = -1
                    print(f"❌ Run {i+1} failed for {model_type} on {compute_pool}: {str(e)}")
                
                # --- FAULT-TOLERANT LOGGING ---
                result_data = [(model_type, compute_pool, i + 1, num_cols_to_use, num_rows_to_use, duration_time, start_time)]
                
                try:
                    results_df = session.create_dataframe(result_data, schema=RESULTS_SCHEMA)
                    results_df.write.mode("append").save_as_table(results_table_name)
                    print(f"✅ Run {i+1}/{runs_to_use} logged for {model_type} on {compute_pool}. Time: {duration_time:.4f}s")
                except Exception as e:
                    print(f"❌ Failed to log results for {model_type} on {compute_pool}: {str(e)}")
                
            return f"Job completed. Successfully processed {runs_to_use} runs for {model_type} on Pool {compute_pool}."

        total_jobs_submitted += 1
        print(f"Submitting Job {total_jobs_submitted}/{total_jobs_in_set}: Model: {MODEL_TYPE_CURRENT} | Pool: {current_pool} | Rows: {num_rows_to_use:,}")

        # Submit the job and WAIT for it to complete before moving to next job
        job = ml_run_remote_job(
            MODEL_TYPE_CURRENT,
            current_pool,
            ESTIMATOR_CLASS,
            ESTIMATOR_PARAMS,
            num_cols_to_use, 
            num_rows_to_use, 
            runs_to_use,
            data_table_name=DATA_TABLE_NAME,
            total_features=NUM_TOTAL_FEATURES,
            results_table_name=RESULTS_TABLE_NAME
        )
        
        # 🔑 KEY CHANGE: Wait for THIS job to complete before submitting the next one
        try:
            print(f"⏳ Waiting for {MODEL_TYPE_CURRENT} on {pool_name} to complete...")
            completion_message = job.wait()
            completed_jobs += 1
            print(f"✅ {MODEL_TYPE_CURRENT} on {pool_name} completed successfully")
        except Exception as e:
            failed_jobs += 1
            print(f"❌ {MODEL_TYPE_CURRENT} on {pool_name} FAILED: {e}")
            # Continue with next job even if this one failed
    
    print(f"🏁 Pool {pool_name} finished: {completed_jobs} successful, {failed_jobs} failed")
    return pool_name, completed_jobs, failed_jobs

# Execute pools in parallel, but jobs within each pool sequentially
with concurrent.futures.ThreadPoolExecutor(max_workers=len(comp_pool_lst)) as executor:
    # Submit all pool processing tasks (each will run its jobs sequentially)
    future_to_pool = {
        executor.submit(process_pool_jobs_sequentially, pool_name, pool_combinations): pool_name
        for pool_name, pool_combinations in combinations_by_pool.items()
    }
    
    # Collect results as they complete
    pool_results = {}
    for future in concurrent.futures.as_completed(future_to_pool):
        pool_name = future_to_pool[future]
        try:
            pool_name, completed, failed = future.result()
            pool_results[pool_name] = {'completed': completed, 'failed': failed}
            print(f"📊 Pool {pool_name} summary: {completed} completed, {failed} failed")
        except Exception as e:
            print(f"💥 Pool {pool_name} processing failed entirely: {e}")
            pool_results[pool_name] = {'completed': 0, 'failed': 'entire_pool_failed'}

# --- Final Output ---
print("\n" + "="*60)
print("--- BENCHMARK COMPLETION SUMMARY ---")
print("="*60)

total_completed = 0
total_failed = 0

for pool_name, results in pool_results.items():
    completed = results['completed']
    failed = results['failed']
    
    if isinstance(failed, int):
        total_completed += completed
        total_failed += failed
        success_rate = (completed / (completed + failed) * 100) if (completed + failed) > 0 else 0
        print(f"🏊 {pool_name:20} | ✅ {completed:3d} | ❌ {failed:3d} | Success: {success_rate:5.1f}%")
    else:
        print(f"💥 {pool_name:20} | ENTIRE POOL FAILED")

print("-" * 60)
print(f"🎯 OVERALL SUMMARY:")
print(f"   Total jobs submitted: {total_jobs_submitted}")
print(f"   Total successful: {total_completed}")
print(f"   Total failed: {total_failed}")
if total_jobs_submitted > 0:
    overall_success_rate = (total_completed / total_jobs_submitted * 100)
    print(f"   Overall success rate: {overall_success_rate:.1f}%")

print(f"\n📊 All results saved to table: {RESULTS_TABLE_NAME}")
print("   (Failed runs logged with duration = -1)")

In [ ]:
# Comprehensive statistical summary of ML_BENCHMARK_RESULTS
import pandas as pd
from scipy import stats
import numpy as np

print("📊 COMPREHENSIVE STATISTICAL SUMMARY - ML_BENCHMARK_RESULTS")
print("=" * 70)

# Load data
df = session.table("ML_BENCHMARK_RESULTS").to_pandas()
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Basic info
print(f"\n🔍 DATA OVERVIEW")
print("-" * 30)
print(df.info())

# Missing values
print(f"\n❌ MISSING VALUES")
print("-" * 30)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_summary[missing_summary['Missing Count'] > 0])
if missing_summary['Missing Count'].sum() == 0:
    print("✅ No missing values found")

# Numerical columns analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(f"\n📈 NUMERICAL VARIABLES SUMMARY")
    print("-" * 40)
    
    # Enhanced describe
    desc = df[numeric_cols].describe()
    
    # Add additional statistics
    for col in numeric_cols:
        if len(df[col].dropna()) > 0:
            desc.loc['mode', col] = df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
            desc.loc['skewness', col] = stats.skew(df[col].dropna())
            desc.loc['kurtosis', col] = stats.kurtosis(df[col].dropna())
            desc.loc['range', col] = df[col].max() - df[col].min()
            desc.loc['iqr', col] = df[col].quantile(0.75) - df[col].quantile(0.25)
    
    print(desc.round(3))

# Failed runs analysis
if 'DURATION_SECONDS' in df.columns:
    print(f"\n⚠️  DURATION ANALYSIS")
    print("-" * 30)
    failed_runs = (df['DURATION_SECONDS'] <= 0).sum()
    success_runs = (df['DURATION_SECONDS'] > 0).sum()
    print(f"Successful runs: {success_runs:,} ({success_runs/len(df)*100:.1f}%)")
    print(f"Failed runs: {failed_runs:,} ({failed_runs/len(df)*100:.1f}%)")
    
    if success_runs > 0:
        valid_durations = df[df['DURATION_SECONDS'] > 0]['DURATION_SECONDS']
        print(f"Duration stats (successful runs only):")
        print(f"  Mean: {valid_durations.mean():.3f}s")
        print(f"  Median: {valid_durations.median():.3f}s") 
        print(f"  Min: {valid_durations.min():.3f}s")
        print(f"  Max: {valid_durations.max():.3f}s")
        print(f"  Std Dev: {valid_durations.std():.3f}s")

# Categorical columns analysis
categorical_cols = df.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print(f"\n📝 CATEGORICAL VARIABLES SUMMARY")
    print("-" * 40)
    
    for col in categorical_cols:
        print(f"\n{col}:")
        value_counts = df[col].value_counts()
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Most frequent: '{value_counts.index[0]}' ({value_counts.iloc[0]:,} times, {value_counts.iloc[0]/len(df)*100:.1f}%)")
        if len(value_counts) > 1:
            print(f"  Least frequent: '{value_counts.index[-1]}' ({value_counts.iloc[-1]:,} times, {value_counts.iloc[-1]/len(df)*100:.1f}%)")
        
        print("  Top 5 values:")
        for idx, (val, count) in enumerate(value_counts.head().items()):
            print(f"    {idx+1}. '{val}': {count:,} ({count/len(df)*100:.1f}%)")

# Correlation analysis for numeric columns
if len(numeric_cols) > 1:
    print(f"\n🔗 CORRELATION MATRIX")
    print("-" * 30)
    corr_matrix = df[numeric_cols].corr()
    print(corr_matrix.round(3))
    
    # Find highest correlations
    print(f"\nHighest correlations:")
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    high_corr = corr_matrix.mask(mask).stack().sort_values(key=abs, ascending=False).head(3)
    for (var1, var2), corr_val in high_corr.items():
        print(f"  {var1} ↔ {var2}: {corr_val:.3f}")

# Distribution analysis
if 'DURATION_SECONDS' in df.columns:
    valid_durations = df[df['DURATION_SECONDS'] > 0]['DURATION_SECONDS']
    if len(valid_durations) > 0:
        print(f"\n📊 DURATION DISTRIBUTION (Successful runs)")
        print("-" * 45)
        
        # Percentiles
        percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
        print("Percentiles:")
        for p in percentiles:
            val = np.percentile(valid_durations, p)
            print(f"  P{p:2d}: {val:8.3f}s")
        
        # Quick histogram
        print(f"\nDuration ranges (successful runs):")
        bins = [0, 1, 5, 10, 30, 60, 300, float('inf')]
        labels = ['<1s', '1-5s', '5-10s', '10-30s', '30-60s', '1-5m', '>5m']
        duration_ranges = pd.cut(valid_durations, bins=bins, labels=labels, right=False)
        range_counts = duration_ranges.value_counts().sort_index()
        for label, count in range_counts.items():
            pct = count / len(valid_durations) * 100
            print(f"  {label:>5}: {count:6,} ({pct:5.1f}%)")

print(f"\n✅ Summary complete!")

In [ ]:
# Two-tiered ML model with fallback for datasets with no failures
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
import numpy as np

print("🎯 Training Two-Tiered ML Model...")
print("=" * 50)

# Load and prep data
results_df = session.table("ML_BENCHMARK_RESULTS").to_pandas()
print(f"Total runs: {len(results_df):,}")

# Check success/failure distribution
success_count = (results_df['DURATION_SECONDS'] > 0).sum()
failure_count = (results_df['DURATION_SECONDS'] <= 0).sum()
print(f"Success runs: {success_count:,}, Failed runs: {failure_count:,}")

# Encode categorical variables
le_model = LabelEncoder()
le_pool = LabelEncoder()
results_df['MODEL_ENCODED'] = le_model.fit_transform(results_df['MODEL_CLASS'])
results_df['POOL_ENCODED'] = le_pool.fit_transform(results_df['COMPUTE_POOL'])

feature_cols = ['MODEL_ENCODED', 'POOL_ENCODED', 'N_COLS_SAMPLED', 'N_ROWS_SAMPLED']
X = results_df[feature_cols]

# Initialize variables
classifier = None
class_accuracy = None

# --- TIER 1: SUCCESS/FAILURE CLASSIFICATION (if needed) ---
if failure_count > 0:
    print("\n🔍 TIER 1: Training Success/Failure Classifier")
    print("-" * 45)
    
    y_success = (results_df['DURATION_SECONDS'] > 0).astype(int)
    success_rate = y_success.mean()
    print(f"Overall success rate: {success_rate:.1%}")
    
    try:
        # Train classification model
        X_train, X_test, y_train_class, y_test_class = train_test_split(
            X, y_success, test_size=0.2, random_state=42, stratify=y_success
        )
        
        classifier = LogisticRegression(random_state=42, max_iter=1000)
        classifier.fit(X_train, y_train_class)
        
        # Evaluate classifier
        y_pred_class = classifier.predict(X_test)
        class_accuracy = accuracy_score(y_test_class, y_pred_class)
        print(f"Classification Accuracy: {class_accuracy:.3f}")
        
    except ValueError as e:
        print(f"⚠️ Classification model failed: {e}")
        print("📝 Assuming all runs will succeed (no failure pattern detected)")
        classifier = None
else:
    print("\n✅ TIER 1: Skipped - No failed runs detected")
    print("📝 All runs successful, assuming 100% success rate")

# --- TIER 2: DURATION REGRESSION ---
print("\n⏱️ TIER 2: Training Duration Regressor")
print("-" * 40)

# Filter to successful runs only for training
success_df = results_df[results_df['DURATION_SECONDS'] > 0].copy()
X_success = success_df[feature_cols]
y_duration = success_df['DURATION_SECONDS']

print(f"Training on {len(success_df):,} successful runs")

# Train regression model
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_success, y_duration, test_size=0.2, random_state=42
)

regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
regressor.fit(X_train_reg, y_train_reg)

# Evaluate regressor
y_pred_reg = regressor.predict(X_test_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
mae = mean_absolute_error(y_test_reg, y_pred_reg)

print(f"Regression R²: {r2:.3f}")
print(f"Regression MAE: {mae:.2f}s")

# --- COMBINED PREDICTION FUNCTION ---
def predict_ml_execution(model_class, compute_pool, n_cols, n_rows, prob_threshold=0.5):
    """
    Two-tiered prediction: First predicts if run will succeed, then predicts duration
    """
    try:
        model_enc = le_model.transform([model_class])[0]
        pool_enc = le_pool.transform([compute_pool])[0]
    except ValueError:
        return {
            'will_succeed': False,
            'success_probability': 0.0,
            'predicted_duration': None,
            'confidence': 'Unknown model/pool combination'
        }
    
    features = np.array([[model_enc, pool_enc, n_cols, n_rows]])
    
    # Tier 1: Predict success probability (if classifier exists)
    if classifier is not None:
        success_prob = classifier.predict_proba(features)[0, 1]
        will_succeed = success_prob >= prob_threshold
    else:
        # No failures in training data, assume success
        success_prob = 1.0
        will_succeed = True
    
    # Tier 2: Predict duration if likely to succeed
    predicted_duration = None
    confidence = "Low"
    
    if will_succeed:
        predicted_duration = regressor.predict(features)[0]
        
        # Confidence based on probability
        if success_prob >= 0.9:
            confidence = "High"
        elif success_prob >= 0.7:
            confidence = "Medium"
        else:
            confidence = "Low"
    
    return {
        'will_succeed': will_succeed,
        'success_probability': success_prob,
        'predicted_duration': predicted_duration,
        'confidence': confidence
    }

# --- TESTING THE COMBINED MODEL ---
print(f"\n🧪 TESTING COMBINED MODEL")
print("-" * 30)

test_cases = [
    ('LogisticRegression', 'CPU_X64_XS_TEST', 25, 50000),
    ('RandomForestClassifier', 'CPU_X64_S_TEST', 50, 250000),
    ('XGBClassifier', 'CPU_X64_M_TEST', 75, 650000),
    ('SVC', 'CPU_X64_SL_TEST', 100, 1000000),
    ('KNeighborsClassifier', 'CPU_X64_XS_TEST', 100, 1000000)
]

for model_class, compute_pool, n_cols, n_rows in test_cases:
    result = predict_ml_execution(model_class, compute_pool, n_cols, n_rows)
    
    print(f"\n{model_class} on {compute_pool}")
    print(f"  Data: {n_cols} cols, {n_rows:,} rows")
    print(f"  Will succeed: {result['will_succeed']} ({result['success_probability']:.1%} confidence)")
    if result['predicted_duration']:
        print(f"  Predicted duration: {result['predicted_duration']:.1f}s")
    print(f"  Confidence: {result['confidence']}")

print(f"\n✅ Two-Tiered Model Complete!")
if class_accuracy:
    print(f"📈 Tier 1 (Success): {class_accuracy:.1%} accuracy")
else:
    print(f"📈 Tier 1 (Success): Skipped - no failures detected")
print(f"📊 Tier 2 (Duration): R²={r2:.3f}, MAE={mae:.1f}s")
print(f"🎯 Use predict_ml_execution() for combined predictions")